In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
%pip install -q optuna scikit-learn pandas pyarrow mlflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.1/51.1 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 442.4/442.4 kB 25.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 59.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.1/4.1 MB 46.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 32.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.7/268.7 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 228.4/228.4 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 136.5/136.5 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1

In [3]:
import sys
sys.path.append('/content/drive/MyDrive/repositories/aq_edge')

In [4]:
%cd /content/drive/MyDrive/repositories/aq_edge/colab
%run mlflow_colab.py

/content/drive/MyDrive/repositories/aq_edge/colab
MLflow server ready: http://127.0.0.1:5000


In [19]:
%cd /content/drive/MyDrive/repositories/aq_edge

import os
os.environ["MLFLOW_TRACKING_URI"] = "http://127.0.0.1:5000"

# Ausführen des Skripts mit expliziter Übergabe der Parameter für die Station USAM und 2 Trials
!python hyperparameter_optuna.py --data.station 'PGB' --optuna.n_trials 1

/content/drive/MyDrive/repositories/aq_edge/colab
[2026-10-21 10:47:44] INFO: Logging to file: /content/drive/MyDrive/repositories/aq_edge/artifacts/preprocessing_20261021_104744.log
[2026-10-21 10:47:44] INFO: Logging to file: /content/drive/MyDrive/repositories/aq_edge/artifacts/metrics_20261021_104744.log
[2026-10-21 10:47:48] INFO: Logging to file: /content/drive/MyDrive/repositories/aq_edge/artifacts/main_20261021_104748.log
INFO:__main__:Logging to file: /content/drive/MyDrive/repositories/aq_edge/artifacts/main_20261021_104748.log
[2026-10-21 10:47:48] INFO: Starting CO2 forecasting hyperparameter optimization
INFO:__main__:Starting CO2 forecasting hyperparameter optimization
[2026-10-21 10:47:48] INFO: Device: cpu
INFO:__main__:Device: cpu
[2026-10-21 10:47:49] INFO: Station: PGB
INFO:__main__:Station: PGB
[2026-10-21 10:47:49] INFO: Features: ['TEM', 'HUM', 'VOC']
INFO:__main__:Features: ['TEM', 'HUM', 'VOC']
[2026-10-21 10:47:49] INFO: Target: CO2
INFO:__main__:Target: CO2
[2

In [20]:
import os
import shutil
import sqlite3

os.makedirs(BACKUP, exist_ok=True)

# Consistent SQLite backup
src = sqlite3.connect(DB)
dst = sqlite3.connect("/content/mlflow_snapshot.db")
with dst:
    src.backup(dst)
src.close()
dst.close()
shutil.copy2("/content/mlflow_snapshot.db", f"{BACKUP}/mlflow.db")

# Preserve experiment artifacts
if os.path.isdir(ARTIFACTS):
    shutil.copytree(
        ARTIFACTS, f"{BACKUP}/mlartifacts", dirs_exist_ok=True
    )

print("MLflow database and artifacts backed up to Drive.")

MLflow database and artifacts backed up to Drive.


In [28]:
import json
from pathlib import Path
import shutil

base_repo_dir = Path("/content/drive/MyDrive/repositories/aq_edge")
station_name = 'PGB'

# Define path to the directory where the run actually created the results
optuna_results_dir = base_repo_dir / "optuna_results"
target_station_dir = optuna_results_dir / station_name

target_station_dir.mkdir(parents=True, exist_ok=True)

# Dynamic pattern lookup using the station_name variable for original output files in the root optuna_results folder
json_sources = list(optuna_results_dir.glob(f"best_co2_{station_name}*.json"))
csv_sources = list(optuna_results_dir.glob(f"trials_co2_{station_name}*.csv"))

# Move original files directly into the station folder
for src in json_sources:
    dest_path = target_station_dir / src.name
    shutil.move(str(src), str(dest_path))

for src in csv_sources:
    dest_path = target_station_dir / src.name
    shutil.move(str(src), str(dest_path))

In [29]:
from pathlib import Path
import shutil

base_repo_dir = Path("/content/drive/MyDrive/repositories/aq_edge")
station_name = 'PGB'

# Recursively find any generated output files under the repository
json_sources = list(base_repo_dir.rglob(f"best_co2_{station_name}*.json"))
csv_sources = list(base_repo_dir.rglob(f"trials_co2_{station_name}*.csv"))

target_station_dir = base_repo_dir / "optuna_results" / station_name
target_station_dir.mkdir(parents=True, exist_ok=True)

print(f"Found JSON files at: {[str(p) for p in json_sources]}")
print(f"Found CSV files at: {[str(p) for p in csv_sources]}")

# Move found files directly to the target folder, avoiding moving a file onto itself
for src in json_sources:
    dest_path = target_station_dir / src.name
    if src.resolve() != dest_path.resolve():
        shutil.move(str(src), str(dest_path))
        print(f"Moved: {src.resolve()} -> {dest_path.resolve()}")

for src in csv_sources:
    dest_path = target_station_dir / src.name
    if src.resolve() != dest_path.resolve():
        shutil.move(str(src), str(dest_path))
        print(f"Moved: {src.resolve()} -> {dest_path.resolve()}")

Found JSON files at: ['/content/drive/MyDrive/repositories/aq_edge/mlflow_backup/mlartifacts/4/5e89fa7667964ffc80511a7d5aabe914/artifacts/optuna/best_co2_PGB_base_lstm_test.json']
Found CSV files at: ['/content/drive/MyDrive/repositories/aq_edge/mlflow_backup/mlartifacts/4/5e89fa7667964ffc80511a7d5aabe914/artifacts/optuna/trials_co2_PGB_base_lstm_test.csv']
Moved: /content/drive/MyDrive/repositories/aq_edge/mlflow_backup/mlartifacts/4/5e89fa7667964ffc80511a7d5aabe914/artifacts/optuna/best_co2_PGB_base_lstm_test.json -> /content/drive/MyDrive/repositories/aq_edge/optuna_results/PGB/best_co2_PGB_base_lstm_test.json
Moved: /content/drive/MyDrive/repositories/aq_edge/mlflow_backup/mlartifacts/4/5e89fa7667964ffc80511a7d5aabe914/artifacts/optuna/trials_co2_PGB_base_lstm_test.csv -> /content/drive/MyDrive/repositories/aq_edge/optuna_results/PGB/trials_co2_PGB_base_lstm_test.csv
